# 열 경계층(수동 스칼라) 검증 — Li, Schlatter, Brandt & Henningson (2009) IJHFF 30, 916

`apps/thermal_boundary_layer` 의 `stats/<case>/bl_stats_*.dat`(속도) 와 `bl_scalar<n>_stats_*.dat`(스칼라) 를 읽어
논문과 같은 통계량으로 비교한다. 조건: Re_δ*₀ 450, 상자 750×34×40 δ*₀, 트립 x=10, 스칼라 θ1..θ5 (Pr 0.2 iso · 0.71 iso · 0.71 isoflux · 2.0 iso · 2.0 isoflux).

참조: **Li 2009** 그림 판독(`plot/ref/li2009_*.csv`, ±3~5%), **Kays–Crawford** St = 0.0125 Pr⁻⁰·⁴ Re_θ⁻⁰·²⁵, **Kader (1981)** θ⁺ 법칙,
**Kasagi/Iida 채널 Re_τ 150 Pr 0.71**(THTLAB 수치표; 벽 근처 y⁺ ≲ 100 보조), 속도장은 KTH DNS(Schlatter & Örlü 2010).

구성 — §1 모니터·전이 · §2 St(Re_θ)·2St/c_f · §3 속도장 점검 · §4 스테이션 Re_θ 830 스칼라 프로파일 · §5 Pr_t · §6 요약 표.

In [ ]:
import glob, os, re
import numpy as np
import matplotlib
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt

APP_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
REF_DIR = os.path.join(os.getcwd(), "ref")
KTH_REF = os.path.join(APP_DIR, "..", "boundary_layer", "plot", "ref")
RE_DSTAR0 = 450.0

CASES = [
    dict(label="DNS 1600x160x192", stats="stats/li2009",        log="logs/tbl_li2009_*.log",        ls="-"),
    dict(label="coarse 1024x112x144", stats="stats/li2009_coarse", log="logs/tbl_li2009_coarse_*.log", ls="--"),
]
STATION = 830.0     # 논문 스칼라 프로파일 (fig.8–10)

# 스칼라 색: Pr 로 색, 벽 조건으로 선 (isoflux 는 점선 마커)
PR_COLOR = {0.2: "#eb6834", 0.71: "#2a78d6", 2.0: "#1baf7a"}
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
                     "axes.edgecolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True,
                     "grid.color": GRID, "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
                     "lines.linewidth": 1.8, "legend.frameon": False, "legend.fontsize": 8})

def header_values(line):
    return {k: float(v) for k, v in re.findall(r"(\w+)=([-\d.eE+]+)", line)}

def read_sections(path):
    """section 1 배열 + section 2 스테이션 목록 (메타 + 행렬)."""
    meta, rows, stations, current = {}, [], [], None
    for line in open(path):
        if line.startswith("# section 2"):
            current = dict(meta=header_values(line), rows=[]); stations.append(current); continue
        if line.startswith("#"):
            if not meta: meta = header_values(line)
            m = re.search(r"wall=(\w+)", line)
            if m: meta["wall"] = m.group(1)
            continue
        if not line.strip(): continue
        values = [float(v) for v in line.split()]
        (rows if current is None else current["rows"]).append(values)
    for st in stations: st["rows"] = np.array(st["rows"])
    return meta, np.array(rows), stations

def pick(folder, pattern):
    files = glob.glob(os.path.join(APP_DIR, folder, pattern))
    if not files: return None
    acc = {p: header_values(open(p).readline()).get("accumulated_time", 0) for p in files}
    return max(files, key=lambda p: (acc[p], os.path.getmtime(p)))

def read_monitor(pattern):
    files = sorted(glob.glob(os.path.join(APP_DIR, pattern)), key=os.path.getmtime)
    if not files: return None
    rows = [[float(v) for v in l.split()] for l in open(files[-1]) if re.match(r"^\s+\d+\s", l) and len(l.split()) == 8]
    d = np.array(rows); return dict(step=d[:, 0], t=d[:, 1], dt=d[:, 2], cf_mid=d[:, 5])

runs = []
for spec in CASES:
    vpath = pick(spec["stats"], "bl_stats_*.dat")
    if vpath is None: print(f"[{spec['label']}] 통계 없음"); continue
    vmeta, vint, vst = read_sections(vpath)
    scalars = []
    for spath in sorted(glob.glob(os.path.join(APP_DIR, spec["stats"], "bl_scalar*_stats_*.dat"))):
        n = int(re.search(r"bl_scalar(\d+)_", spath).group(1))
        if spath != pick(spec["stats"], f"bl_scalar{n}_stats_*.dat"): continue
        smeta, sint, sst = read_sections(spath)
        scalars.append(dict(n=n, pr=smeta["Pr"], wall=smeta["wall"], meta=smeta, integ=sint, stations=sst,
                            color=PR_COLOR.get(round(smeta["Pr"], 2), "#4a3aa7"), ls=":" if smeta["wall"] == "isoflux" else "-"))
    runs.append(dict(spec, vmeta=vmeta, vint=vint, vst=vst, scalars=scalars, monitor=read_monitor(spec["log"])))
    print(f"[{spec['label']}] {os.path.basename(vpath)} t={vmeta['t']:.0f} 누적 {vmeta['accumulated_time']:.0f}; 스칼라 {len(scalars)}개: "
          + ", ".join(f"θ{s['n']} Pr {s['pr']} {s['wall']}" for s in scalars))

# ── 참조 ──────────────────────────────────────────────────────────────────
def read_csv(name):
    lines = [l for l in open(os.path.join(REF_DIR, name)) if l.strip() and not l.startswith("#")]
    names = lines[0].strip().split(","); rows = [l.strip().split(",") for l in lines[1:]]
    out = {}
    for i, n in enumerate(names):
        try: out[n] = np.array([float(r[i]) for r in rows])
        except ValueError: out[n] = [r[i] for r in rows]
    return out
li_st = read_csv("li2009_stanton.csv"); li_prof = read_csv("li2009_profiles_re830.csv")
KC = lambda pr, re_theta: 0.0125 * pr ** -0.4 * re_theta ** -0.25
def kader(pr, yp):
    beta = (3.85 * pr ** (1 / 3) - 1.3) ** 2 + 2.12 * np.log(pr)
    gamma = 0.01 * (pr * yp) ** 4 / (1 + 5 * pr ** 3 * yp)
    return pr * yp * np.exp(-gamma) + (2.12 * np.log(1 + yp) + beta) * np.exp(-1 / gamma)
# Kasagi/Iida 채널 (THTLAB CH122_PG.WL4): 블록 1 = y+ u+ t+ urms vrms wrms trms prms, 블록 2 = y+ -uv+ ut+ -vt+
def read_kasagi(path):
    blocks, current = [], None
    for line in open(path):
        if re.match(r"^\s*y\+\s", line): current = []; blocks.append(current); continue
        if current is not None:
            parts = line.split()
            if parts and re.match(r"^-?\d", parts[0]):
                try: current.append([float(v) for v in parts])
                except ValueError: pass
            elif current and not line.strip(): pass
    blocks = [b for b in blocks if len(b) > 5]          # 명명법의 "y+ = ..." 줄이 만든 빈 블록 제거
    b1, b2 = np.array(blocks[0]), np.array(blocks[1])
    return dict(yp=b1[:, 0], Up=b1[:, 1], Tp=b1[:, 2], trms=b1[:, 6], yp2=b2[:, 0], uv=-b2[:, 1], ut=b2[:, 2], vt=-b2[:, 3])
kasagi = read_kasagi(os.path.join(REF_DIR, "kasagi_iida2001_ch150_pr071.txt"))
kth = {}
for f in sorted(glob.glob(os.path.join(KTH_REF, "vel_*_dns.prof"))):
    h = open(f).read(); g = lambda k: float(re.search(k + r"\s*=\s*([0-9.eE+-]+)", h).group(1))
    kth[round(g(r"Re_\{\\theta\}"))] = dict(re_theta=g(r"Re_\{\\theta\}"), cf=g(r"c_f"), H=g(r"H_\{12\}"), re_tau=g(r"Re_\{\\tau\}"))
print("KTH velocity references:", sorted(kth))

## 1. 모니터 — 전이와 시간 간격

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for r in runs:
    m = r["monitor"]
    if m is None: continue
    axes[0].plot(m["t"], m["cf_mid"], ls=r["ls"], color="#2a78d6", label=r["label"])
    axes[1].plot(m["t"], m["dt"], ls=r["ls"], color="#2a78d6", label=r["label"])
axes[0].set_ylabel("C_f at x = L_x/2"); axes[1].set_ylabel("dt"); axes[1].set_yscale("log")
for ax in axes: ax.set_xlabel("t  (delta*_0/U_inf)"); ax.legend()
figure.tight_layout(); plt.show()

## 2. Stanton 수 St(Re_θ) — 논문 fig.4 와 Kays–Crawford

실선 = 우리(isoscalar), 점선 = 우리(isoflux); 기호 = Li 2009 판독; 가는 검은 선 = Kays–Crawford. 오른쪽: 레이놀즈 상사 계수 2St/c_f.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for r in runs:
    for s in r["scalars"]:
        x, rt, st, ra = s["integ"][:, 0], s["integ"][:, 1], s["integ"][:, 2], s["integ"][:, 7]
        ok = (x > 30) & (x < 0.95 * x.max()) & (rt > 200)
        axes[0].plot(rt[ok], st[ok], color=s["color"], ls=s["ls"], label=f"{r['label']} theta{s['n']} Pr {s['pr']} {s['wall']}" if r is runs[0] else None,
                     lw=1.8 if r is runs[0] else 1.0)
        axes[1].plot(rt[ok], ra[ok], color=s["color"], ls=s["ls"], lw=1.8 if r is runs[0] else 1.0)
for key, pr, marker in (("St_pr071_iso", 0.71, "o"), ("St_pr071_flux", 0.71, "s"), ("St_pr02_iso", 0.2, "^"), ("St_pr2_iso", 2.0, "v")):
    axes[0].scatter(li_st["Re_theta"], li_st[key], marker=marker, s=34, facecolor="white", edgecolor=PR_COLOR[pr], zorder=5,
                    label=f"Li 2009 {key.replace('St_', '')}")
rt = np.linspace(250, 900, 50)
for pr in (0.2, 0.71, 2.0): axes[0].plot(rt, KC(pr, rt), color=INK, lw=0.8, ls="-", alpha=0.5)
axes[0].set_xlim(200, 900); axes[0].set_ylim(1e-3, 6e-3); axes[0].set_xlabel("Re_theta"); axes[0].set_ylabel("St"); axes[0].legend(fontsize=6.5, ncol=2)
axes[0].set_title("Stanton number (thin black = Kays-Crawford 0.0125 Pr^-0.4 Re_theta^-0.25)")
axes[1].set_xlim(200, 900); axes[1].set_ylim(0.5, 2.0); axes[1].set_xlabel("Re_theta"); axes[1].set_ylabel("2 St / C_f"); axes[1].set_title("Reynolds analogy factor")
figure.tight_layout(); plt.show()

## 3. 속도장 점검 — c_f, H (KTH DNS, Österlund 상관식)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(11, 3.8))
rt = np.linspace(250, 900, 50)
axes[0].plot(rt, 2 * (np.log(rt) / 0.384 + 3.75) ** -2, color=INK, lw=0.8, ls=":", label="Osterlund (Li eq.14)")
for r in runs:
    v = r["vint"]; x, rt_v = v[:, 0], v[:, 7]; ok = (x > 30) & (x < 0.95 * x.max()) & (rt_v > 200)
    axes[0].plot(rt_v[ok], v[ok, 2], ls=r["ls"], color="#2a78d6", label=r["label"]); axes[1].plot(rt_v[ok], v[ok, 6], ls=r["ls"], color="#2a78d6")
for key, ref in kth.items():
    axes[0].scatter(ref["re_theta"], ref["cf"], marker="o", s=36, facecolor="white", edgecolor=INK, zorder=5, label="KTH DNS" if key == min(kth) else None)
    axes[1].scatter(ref["re_theta"], ref["H"], marker="o", s=36, facecolor="white", edgecolor=INK, zorder=5)
axes[0].set_xlim(200, 900); axes[0].set_ylabel("C_f"); axes[1].set_xlim(200, 900); axes[1].set_ylabel("H"); axes[1].set_ylim(1.35, 1.7)
for ax in axes: ax.set_xlabel("Re_theta")
axes[0].legend(fontsize=8); figure.tight_layout(); plt.show()

## 4. 스테이션 Re_θ ≈ 830 — 평균 스칼라·변동·플럭스 (논문 fig.8–9)

(a) θ⁺ 와 Kader 법칙, (b) 진단함수 Ξ_θ = y⁺ dθ⁺/dy⁺ (1/κ_θ = 2.44 선), (c) θ_rms⁺ (기호 = Li 정점·벽값, 회색 = Kasagi 채널 Pr 0.71), (d) ⟨u′θ′⟩⁺/Pr, (e) −⟨w′θ′⟩⁺/Pr (w = 벽수직 = 논문 v).

In [ ]:
def station(run, target):
    vst = min(run["vst"], key=lambda s: abs(s["meta"]["Re_theta"] - target))
    out = dict(v=vst, s={})
    for s in run["scalars"]:
        st = min(s["stations"], key=lambda q: abs(q["meta"]["Re_theta"] - target))
        out["s"][s["n"]] = (s, st)
    return out
main = runs[0]; st830 = station(main, STATION)
re_here = st830["v"]["meta"]["Re_theta"]
figure, axes = plt.subplots(1, 5, figsize=(22, 4.2))
yl = np.logspace(-1, 3, 200)
for n, (s, st) in st830["s"].items():
    d = st["rows"]; yp, tp, trms, ut, wt = d[:, 1], d[:, 2], d[:, 3], d[:, 4], d[:, 5]
    lab = f"theta{n} Pr {s['pr']} {s['wall']}"
    axes[0].semilogx(yp, tp, color=s["color"], ls=s["ls"], label=lab)
    good = (yp > 1) & (yp < 600)
    xi = yp[good][1:-1] * np.gradient(tp[good], np.log(yp[good]))[1:-1] / yp[good][1:-1]
    axes[1].semilogx(yp[good][1:-1], yp[good][1:-1] * np.gradient(tp[good], yp[good])[1:-1], color=s["color"], ls=s["ls"])
    axes[2].semilogx(yp, trms, color=s["color"], ls=s["ls"])
    axes[3].loglog(yp, np.abs(ut) / s["pr"], color=s["color"], ls=s["ls"])
    axes[4].loglog(yp, np.abs(wt) / s["pr"], color=s["color"], ls=s["ls"])
for pr in (0.2, 0.71, 2.0):
    axes[0].semilogx(yl, kader(pr, yl), color=PR_COLOR[pr], lw=0.8, ls="-.", alpha=0.7, label=f"Kader Pr {pr}")
axes[1].axhline(1 / 0.41, color=INK, lw=0.8); axes[1].set_ylim(0, 10)
axes[2].semilogx(kasagi["yp"], kasagi["trms"], color="0.55", lw=2.2, alpha=0.6, label="Kasagi/Iida ch150 Pr 0.71")
for i in range(len(li_prof["scalar"])):
    pr = li_prof["Pr"][i]; c = PR_COLOR[round(pr, 2)]; mk = "s" if li_prof["wall"][i] == "isoflux" else "o"
    axes[2].scatter([li_prof["yplus_peak"][i]], [li_prof["theta_rms_peak"][i]], marker=mk, s=40, facecolor="white", edgecolor=c, zorder=5)
    if li_prof["theta_rms_wall"][i] > 0: axes[2].scatter([0.15], [li_prof["theta_rms_wall"][i]], marker=mk, s=40, facecolor="white", edgecolor=c, zorder=5)
axes[3].loglog(kasagi["yp2"], kasagi["ut"] / 0.71, color="0.55", lw=2.2, alpha=0.6); axes[4].loglog(kasagi["yp2"], kasagi["vt"] / 0.71, color="0.55", lw=2.2, alpha=0.6)
axes[0].set_ylim(0, 25); axes[0].set_ylabel("theta+"); axes[0].set_title(f"(a) mean scalar, Re_theta {re_here:.0f}"); axes[0].legend(fontsize=6.5)
axes[1].set_ylabel("y+ dtheta+/dy+"); axes[1].set_title("(b) diagnostic function (line: 1/0.41)")
axes[2].set_ylim(0, 5); axes[2].set_ylabel("theta_rms+"); axes[2].set_title("(c) scalar rms (symbols: Li peak / wall value)"); axes[2].legend(fontsize=7)
axes[3].set_ylim(1e-4, 30); axes[3].set_ylabel("<u'theta'>+ / Pr"); axes[3].set_title("(d) streamwise flux (grey: Kasagi)")
axes[4].set_ylim(1e-6, 3); axes[4].set_ylabel("-<w'theta'>+ / Pr"); axes[4].set_title("(e) wall-normal flux (grey: Kasagi)")
for ax in axes: ax.set_xlim(0.1, 1e3); ax.set_xlabel("y+")
figure.tight_layout(); plt.show()

## 5. 난류 Prandtl 수 Pr_t (논문 fig.10) — 속도·스칼라 스테이션 합성

Pr_t = (⟨u′w′⟩ dθ/dz)/(⟨w′θ′⟩ dU/dz) (벽 단위로 무차원). 논문 Pr 0.71: 벽 ≈1.05, 정점 1.07 (y⁺≈45), 최소 0.72 (y⁺≈220), 가장자리 2차 정점.

In [ ]:
v = st830["v"]["rows"]; yp_v, Up, uw = v[:, 1], v[:, 2], v[:, 6]
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
dUdy = np.gradient(Up, yp_v)
for n, (s, st) in st830["s"].items():
    d = st["rows"]; tp, wt = d[:, 2], d[:, 5]
    dTdy = np.gradient(tp, yp_v)
    prt = (uw * dTdy) / (wt * dUdy + 1e-300)
    ok = (yp_v > 0.3) & (yp_v < 450) & np.isfinite(prt)
    axes[0].plot(yp_v[ok], prt[ok], color=s["color"], ls=s["ls"], label=f"theta{n} Pr {s['pr']} {s['wall']}")
    axes[1].semilogx(yp_v[ok], prt[ok] / (s["pr"] ** 0.4 if s["wall"] == "isoflux" else 1.0), color=s["color"], ls=s["ls"])
for i in range(len(li_prof["scalar"])):
    if li_prof["wall"][i] != "isoscalar": continue
    c = PR_COLOR[round(li_prof["Pr"][i], 2)]
    axes[0].scatter([2, 45, 220], [li_prof["Prt_wall"][i], li_prof["Prt_peak"][i], li_prof["Prt_min"][i]], marker="D", s=30, facecolor="white", edgecolor=c, zorder=5)
axes[0].set_xlim(0, 450); axes[0].set_ylim(0, 1.5); axes[0].set_xlabel("y+"); axes[0].set_ylabel("Pr_t"); axes[0].legend(fontsize=7); axes[0].set_title("(a) Pr_t (diamonds: Li 2009 wall / peak / minimum)")
axes[1].set_xlim(0.3, 500); axes[1].set_ylim(1e-2, 10); axes[1].set_yscale("log"); axes[1].set_xlabel("y+"); axes[1].set_ylabel("Pr_t (isoflux: / Pr^0.4)"); axes[1].set_title("(b) near-wall limit (isoscalar const, isoflux ~ y+)")
figure.tight_layout(); plt.show()

## 6. 요약 표 — Re_θ 830 스테이션과 St

In [ ]:
print(f"station Re_theta {re_here:.1f}  (Re_tau {st830['v']['meta']['Re_tau']:.1f}, Cf {st830['v']['meta']['Cf']:.5f}, H {st830['v']['meta']['H']:.4f})")
print(f"{'scalar':<8} {'Pr':>5} {'wall':<9} {'St':>9} {'Li fig4':>8} {'KC':>8} {'2St/Cf':>7} | {'rms peak':>8} {'@y+':>5} {'Li':>5} {'@y+':>4} | {'rms wall':>8} {'Li':>5} | {'-wt+ y+30-100':>13} {'Li':>5} | {'Prt wall/peak/min':>18} {'Li':>14} | {'d99th+':>7} {'Li':>5}")
for n, (s, st) in st830["s"].items():
    d = st["rows"]; yp, trms, wt = d[:, 1], d[:, 3], d[:, 5]
    m = st["meta"]; pr = s["pr"]
    li_row = next((i for i in range(len(li_prof["scalar"])) if abs(li_prof["Pr"][i] - pr) < 0.05 and li_prof["wall"][i] == s["wall"]), None)
    li_st_key = {(0.71, "isoscalar"): "St_pr071_iso", (0.71, "isoflux"): "St_pr071_flux", (0.2, "isoscalar"): "St_pr02_iso", (2.0, "isoscalar"): "St_pr2_iso"}.get((round(pr, 2), s["wall"]))
    li_st_val = np.interp(re_here, li_st["Re_theta"], li_st[li_st_key]) if li_st_key else float("nan")
    peak_i = int(np.argmax(trms[yp < 300])); plateau = -np.mean(wt[(yp > 30) & (yp < 100)]) / pr
    v = st830["v"]["rows"]; dUdy = np.gradient(v[:, 2], v[:, 1]); dTdy = np.gradient(d[:, 2], v[:, 1]); prt = (v[:, 6] * dTdy) / (wt * dUdy + 1e-300)
    at = lambda y: float(np.interp(y, v[:, 1], prt))
    d99p = m["delta99_theta"] * m["u_tau"] * RE_DSTAR0
    li = (lambda k: li_prof[k][li_row]) if li_row is not None else (lambda k: float("nan"))
    print(f"theta{n:<3} {pr:5.2f} {s['wall']:<9} {m['St']:9.3e} {li_st_val:8.2e} {KC(pr, re_here):8.2e} {2*m['St']/st830['v']['meta']['Cf']:7.3f} | "
          f"{trms[peak_i]:8.2f} {yp[peak_i]:5.0f} {li('theta_rms_peak'):5.2f} {li('yplus_peak'):4.0f} | {trms[0]:8.2f} {li('theta_rms_wall'):5.2f} | "
          f"{plateau:13.2f} {li('vt_plus_plateau'):5.2f} | {at(2):5.2f}/{at(45):5.2f}/{at(220):5.2f} {li('Prt_wall'):4.2f}/{li('Prt_peak'):4.2f}/{li('Prt_min'):4.2f} | {d99p:7.0f} {li('delta99_theta_plus'):5.0f}")
print("velocity delta99+ at this station:", f"{st830['v']['meta']['delta99'] * st830['v']['meta']['u_tau'] * RE_DSTAR0:.0f} (Li: 315)")